In [2]:
from torch.utils.data import DataLoader, ConcatDataset
from h5dataset import h5set
import torch
from torch import nn
import os


# we could also have been used a set seed and have random split be deterministic
dir_path = "/mnt/data/train_test_val"
h5_path = "../data/H1_rechunked.h5"
dir = os.listdir(dir_path)
if len(dir)==0:
    noise = h5set(h5_path, dataset='noise')
    train, val, test = torch.utils.data.random_split(noise, [225000, 75000, 25000])
    injection = h5set(h5_path, dataset='injection')
    test = ConcatDataset([test, injection])
    torch.save(train, "/mnt/data/train_test_val/train.pt") #just saves indices
    torch.save(val, "/mnt/data/train_test_val/val.pt")
    torch.save(test, "/mnt/data/train_test_val/test.pt")
else:
    train = torch.load("/mnt/data/train_test_val/train.pt", weights_only=False)
    val = torch.load("/mnt/data/train_test_val/val.pt", weights_only=False)
    test = torch.load("/mnt/data/train_test_val/test.pt", weights_only=False)


training = DataLoader(train,
                batch_size=1000,
                shuffle=True,
                num_workers=0,
                #persistent_workers=True,
                drop_last=True,
                )




In [19]:
#input_size = n° features ->> 1, deve essere uguale al # di colonne
#hidden size = n° neurons in hidden layer
# num layers = 1
# L = len seq = 16384
input = torch.randn((16384,1)) # gut, h0 e c0 inizializzati a 0 di default
L1 = input.size()[0]  # 16384
h_in1 = input.size()[1]  # 1
h_cell1 = 32
h_out1 = h_cell1
num_layers1 = 1
rnn1 = nn.LSTM(input_size=h_in1, hidden_size=h_cell1, num_layers=num_layers1)

output, (a,b) = rnn1(input)
print("outpuT:::",output.shape)  # (seq_len,  hidden_size)
print(a.shape)     # (num_layers , hidden_size)
print(b.shape)     # (num_layers ,  hidden_size)

rnn2 = nn.LSTM(input_size=h_cell1, hidden_size=1, num_layers=8) #should divide
output2, (hn2, cn2) = rnn2(output)

print("outpuT:::",output2.shape)  # (seq_len, batch, num_directions * hidden_size)
print(hn2.shape)      # (num_layers * num_directions, batch, hidden_size)
print(cn2.shape)      # (num_layers * num_directions, batch,

outpuT::: torch.Size([16384, 32])
torch.Size([1, 32])
torch.Size([1, 32])
outpuT::: torch.Size([16384, 1])
torch.Size([8, 1])
torch.Size([8, 1])


In [ ]:
input = torch.randn((16384,1)) # gut, h0 e c0 inizializzati a 0 di default
input.size()[0]

16384

In [ ]:
class LSTM_encoder(nn.Module):
    super().__init__()

    #LSTM section
    self.lstm1 = nn.LSTM(input_size=1,
                        hidden_size=100,
                        num_layers=1,
                        bias = False,
                        batch_first = False,
                        dropout = 0,
                        bidirectional = False
                        )
    self.lstm2 = nn.LSTM(input_size=32,
                        hidden_size=8,
                        num_layers=1,
                        batch_first = False,
                        dropout = 0,
                        bidirectional = False
                        )

In [ ]:
####moreno
from keras.layers import Input, Dense, LSTM, TimeDistributed, RepeatVector, Conv1D, \
    MaxPooling1D, UpSampling1D, Flatten, Reshape, GRU
from keras.models import Model
from keras import regularizers

def autoencoder_LSTM(X):
    inputs = Input(shape=(X.shape[1], X.shape[2]))
    L1 = LSTM(32, activation='tanh', return_sequences=True, 
              kernel_regularizer=regularizers.l2(0.00))(inputs)
    L2 = LSTM(8, activation='tanh', return_sequences=False)(L1)
    L3 = RepeatVector(X.shape[1])(L2)
    L4 = LSTM(8, activation='tanh', return_sequences=True)(L3)
    L5 = LSTM(32, activation='tanh', return_sequences=True)(L4)
    output = TimeDistributed(Dense(X.shape[2]))(L5)    
    model = Model(inputs=inputs, outputs=output)
    return model